# DS-A3: Regression Baseline and Diagnostics

A reproducible, simple regression study on the supplied OMNI daily dataset. The notebook uses a chronological split and reports only metrics produced by execution.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
DATA = Path('data/omni_01_av.txt')
if not DATA.exists(): DATA = Path('omni_01_av(1).txt')
print('Dataset path:', DATA.resolve())

## Data, target, and limitations
The target is daily solar-wind speed (OMNI2 positional column 34, zero-based index 33). Predictors are proton density, IMF magnitude, GSM Bz, and flow pressure. Fill values are removed; the chronological split is intended to reduce temporal leakage.

In [ ]:
names = ['year', 'day_of_year', 'column_2', 'column_3', 'column_4', 'column_5', 'column_6', 'column_7', 'column_8', 'column_9', 'column_10', 'column_11', 'column_12', 'column_13', 'imf_magnitude', 'column_15', 'column_16', 'column_17', 'proton_density', 'bz_gsm', 'column_20', 'column_21', 'column_22', 'flow_pressure', 'solar_wind_speed', 'column_25', 'column_26', 'column_27', 'column_28', 'column_29', 'column_30', 'column_31', 'column_32', 'column_33', 'column_34', 'column_35', 'column_36', 'column_37', 'column_38', 'column_39', 'column_40', 'column_41', 'column_42', 'column_43', 'column_44', 'column_45', 'column_46', 'column_47', 'column_48', 'column_49', 'column_50', 'column_51', 'column_52', 'column_53', 'column_54']
raw = pd.read_csv(DATA, sep=r'\s+', header=None, names=names, engine='python')
assert raw.shape[1] == 55
clean = raw.apply(pd.to_numeric, errors='coerce')
sentinels = [999.9, 999.99, 9999., 99999., 999999., 9999999., 999999.99, 99999.99, 9999999.99, 99., 999, 9999, 99999, 999999, 9999999]
clean = clean.mask(clean.isin(sentinels))
features = ['proton_density','imf_magnitude','bz_gsm','flow_pressure']; target='solar_wind_speed'
data = clean[['year','day_of_year',target]+features].dropna().copy()
data['date'] = pd.to_datetime(data.year.astype(int).astype(str), format='%Y') + pd.to_timedelta(data.day_of_year.astype(int)-1, unit='D')
data = data.sort_values('date').drop_duplicates('date')
data = data[(data[target]>=100)&(data[target]<=2000)]
for c in features: data = data[data[c].abs()<1e6]
n=len(data); train=data.iloc[:int(.70*n)]; val=data.iloc[int(.70*n):int(.85*n)]; test=data.iloc[int(.85*n):]
print({'raw_rows':len(raw),'usable_rows':len(data),'train':len(train),'validation':len(val),'test':len(test)})

## Baselines and candidate models
The median-only dummy is the naive baseline. Linear, ridge, and lasso models share median imputation and standardization. No hyperparameter search uses the test set.

In [ ]:
models={'Naive median':DummyRegressor(strategy='median'),'Linear regression':make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LinearRegression()),'Ridge (alpha=1)':make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0)),'Lasso (alpha=0.01)':make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Lasso(alpha=.01,max_iter=10000))}
rows=[]; fitted={}
for name, model in models.items():
 model.fit(train[features],train[target]); fitted[name]=model
 for split, part in [('validation',val),('test',test)]:
  p=model.predict(part[features]); rows.append({'model':name,'split':split,'n':len(part),'MAE':mean_absolute_error(part[target],p),'RMSE':mean_squared_error(part[target],p)**.5,'R2':r2_score(part[target],p)})
metrics=pd.DataFrame(rows); display(metrics.round(3))
best_name=metrics[metrics.split=='validation'].sort_values('RMSE').iloc[0]['model']; print('Selected by validation RMSE:',best_name)

## Diagnostics and critical slices
Residuals are observed minus predicted. These plots are diagnostic rather than proof of a causal relationship. A time-based split may expose distribution shift.

In [ ]:
import matplotlib.pyplot as plt
model=fitted[best_name]; pred=model.predict(test[features]); residual=test[target].to_numpy()-pred
fig,ax=plt.subplots(figsize=(6,4)); ax.scatter(pred,residual,s=8,alpha=.35); ax.axhline(0,ls='--'); ax.set(xlabel='Predicted speed (km/s)',ylabel='Residual (km/s)',title='Residuals vs predictions'); plt.show()
fig,ax=plt.subplots(figsize=(6,4)); ax.hist(residual,bins=40); ax.set(xlabel='Residual (km/s)',ylabel='Count',title='Residual distribution'); plt.show()
errors=pd.DataFrame({'prediction':pred,'residual':residual,'absolute_error':np.abs(residual)})
errors['prediction_bin']=pd.qcut(errors.prediction,q=5,duplicates='drop')
display(errors.groupby('prediction_bin',observed=True).agg(n=('residual','size'),mean_residual=('residual','mean'),MAE=('absolute_error','mean'),residual_sd=('residual','std')).round(3))